**Overfitting**

In machine learning, overfitting occurs when an algorithm fits too closely or even exactly to its training data, resulting in a model that can’t make accurate predictions or conclusions from any data other than the training data.

Overfitting defeats purpose of the machine learning model.

**Avoiding Overfitting**

L1 and L2 Regularization

Dropout

**L1 and L2 Regularization**

L1 Regularization: Sets weights close to zero exactly to zero, effectively removing less important features.

L2 Regularization: Reduces large weight values, preventing the model from relying too much on specific features.

Result: Regularization helps prevent overfitting by simplifying the model and improving its generalization ability.

The code aims to reduce overfitting using L2 regularization (shrinkage) while training a deep neural network for image classification.

In [2]:
import tensorflow as tf
from tensorflow import keras
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
pixel_means = X_train.mean(axis=0, keepdims=True)
pixel_stds = X_train.std(axis=0, keepdims=True)
X_train_scaled = (X_train - pixel_means) / pixel_stds
X_valid_scaled = (X_valid - pixel_means) / pixel_stds
X_test_scaled = (X_test - pixel_means) / pixel_stds

In [4]:
layer = keras.layers.Dense(100, activation="elu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=keras.regularizers.l2(0.01))

In [5]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="elu",
                       kernel_initializer="he_normal",
                       kernel_regularizer=keras.regularizers.l2(0.01)),
    keras.layers.Dense(100, activation="elu",
                       kernel_initializer="he_normal",
                       kernel_regularizer=keras.regularizers.l2(0.01)),
    keras.layers.Dense(10, activation="softmax",
                       kernel_regularizer=keras.regularizers.l2(0.01))
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
n_epochs = 2
history = model.fit(X_train_scaled, y_train, epochs=n_epochs,
                    validation_data=(X_valid_scaled, y_valid))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/2
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.7957 - loss: 3.2483 - val_accuracy: 0.8320 - val_loss: 0.7151
Epoch 2/2
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.8225 - loss: 0.7300 - val_accuracy: 0.8404 - val_loss: 0.6847


In [6]:
# third part
from functools import partial

RegularizedDense = partial(keras.layers.Dense,
                           activation="elu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=keras.regularizers.l2(0.01))

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    RegularizedDense(300),
    RegularizedDense(100),
    RegularizedDense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
n_epochs = 2
history = model.fit(X_train_scaled, y_train, epochs=n_epochs,
                    validation_data=(X_valid_scaled, y_valid))

Epoch 1/2
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.7952 - loss: 3.3528 - val_accuracy: 0.8352 - val_loss: 0.7144
Epoch 2/2
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.8248 - loss: 0.7239 - val_accuracy: 0.8262 - val_loss: 0.7151


**Results Analysis**

The third part achieved lower training loss (0.7239 vs. 0.7300) but lower validation accuracy (82.62% vs. 84.04%), indicating that L2 regularization helped prevent overfitting, but further fine-tuning may be needed for optimal performance.



---
**Dropout**

Dropout is a regularization technique where a random subset of neurons is ignored (set to zero) during training, preventing reliance on specific neurons.


For each training batch, randomly "drop" a fraction of nodes in the network.
Ensures the network learns robust patterns rather than memorizing the data.



👩🏻‍🔬 **Experiment: Comparing Two Models – With and Without Dropout**

---
**Overall Purpose of the Code**

The code compares two models to analyze the impact of Dropout on reducing overfitting in a deep neural network.


In [7]:
# First Model (Without Dropout)
# A deep neural network for image classification, trained without Dropout, which increases the risk of overfitting.
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    #keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
    #keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    #keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
n_epochs = 20
history = model.fit(X_train_scaled, y_train, epochs=n_epochs,
                    validation_data=(X_valid_scaled, y_valid))

Epoch 1/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.8078 - loss: 0.5626 - val_accuracy: 0.8700 - val_loss: 0.3641
Epoch 2/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8792 - loss: 0.3309 - val_accuracy: 0.8750 - val_loss: 0.3374
Epoch 3/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.8939 - loss: 0.2838 - val_accuracy: 0.8822 - val_loss: 0.3282
Epoch 4/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9034 - loss: 0.2557 - val_accuracy: 0.8812 - val_loss: 0.3468
Epoch 5/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9132 - loss: 0.2328 - val_accuracy: 0.8798 - val_loss: 0.3466
Epoch 6/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.9186 - loss: 0.2149 - val_accuracy: 0.8830 - val_loss: 0.3424
Epoch 7/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9278 - loss: 0.1939 - val_accuracy: 0.8868 - val_loss: 0.3561
Epoch 8/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.9322 - loss: 0

In [8]:
# Second Model (With Dropout)
# The same network but with Dropout (rate = 0.2) applied after each layer to reduce overfitting by randomly deactivating neurons during training
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
n_epochs = 20
history = model.fit(X_train_scaled, y_train, epochs=n_epochs,
                    validation_data=(X_valid_scaled, y_valid))

Epoch 1/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.7571 - loss: 0.7476 - val_accuracy: 0.8596 - val_loss: 0.3758
Epoch 2/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.8429 - loss: 0.4246 - val_accuracy: 0.8664 - val_loss: 0.3665
Epoch 3/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.8535 - loss: 0.3893 - val_accuracy: 0.8700 - val_loss: 0.3433
Epoch 4/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.8631 - loss: 0.3692 - val_accuracy: 0.8864 - val_loss: 0.3187
Epoch 5/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.8675 - loss: 0.3646 - val_accuracy: 0.8828 - val_loss: 0.3200
Epoch 6/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.8735 - loss: 0.3410 - val_accuracy: 0.8832 - val_loss: 0.3177
Epoch 7/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.8734 - loss: 0.3378 - val_accuracy: 0.8834 - val_loss: 0.3180
Epoch 8/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.8771 - loss: 0

𓂃🖊 **Results Analysis**

Model	| Accuracy |	Loss |	Val Accuracy |	Val Loss


---



Without 	| 96.86% |	0.0855 |	88.32%	|  0.5906

Dropout


---
With | 	89.24% |	0.2834 |	89.34% |	0.3098

Dropout


---

The first model (without Dropout) has very high training accuracy (96.86%) but poor generalization (validation accuracy: 88.32%), indicating overfitting.

The second model (with Dropout) has lower training accuracy (89.24%) but better validation accuracy (89.34%) with a significantly lower validation loss (0.3098 vs. 0.5906), showing that **Dropout improved generalization**
